In [1]:
import pandas as pd

disruptions_df = pd.read_csv('final_feature_df.csv')

In [2]:
import numpy as np

In [3]:
disruptions_df

,Date,source,destination,source_degree,source_weighted_degree,source_avg_distance,target_degree,target_weighted_degree,target_avg_distance,common_neighbors,...,Disrupted,Days_since_last_disruption,Num_prev_disruptions,Ratio,Total_disruptions,Total_rides,Previous_causes_vec,Most_recent_causes_vec,Date.1,Season
0,2019-01-01,'s-Hertogenbosch,Roosendaal,14,83,31.714286,13,52,39.230769,2,...,True,0,0,0.000000,1,2,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,2019-01-01,Winter
1,2019-01-01,'s-Hertogenbosch,Utrecht Centraal,14,83,31.714286,36,243,21.305556,2,...,True,0,0,0.000000,2,1,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. ...,2019-01-01,Winter
2,2019-01-01,Den Haag Centraal,Eindhoven Centraal,29,253,52.793103,21,116,40.142857,3,...,True,0,0,0.000000,1,32,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,2019-01-01,Winter
3,2019-01-01,Den Haag Centraal,Utrecht Centraal,29,253,52.793103,36,243,21.305556,10,...,True,0,0,0.000000,1,21,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,2019-01-01,Winter
4,2019-01-01,Eindhoven Centraal,Den Haag Centraal,21,116,40.142857,29,253,52.793103,3,...,True,0,0,0.000000,1,32,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,2019-01-01,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
575148,2024-12-31,Zwolle,Lelystad Centrum,23,104,53.608696,15,78,59.133333,4,...,False,16,41,0.807882,41,5075,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,2024-12-31,Winter
575149,2024-12-31,Zwolle,Nijmegen,23,104,53.608696,19,64,56.000000,7,...,False,34,29,0.975774,29,2972,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,2024-12-31,Winter
575150,2024-12-31,Zwolle,Roosendaal,23,104,53.608696,18,83,53.277778,4,...,False,9,380,0.595229,380,63841,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,2024-12-31,Winter
575151,2024-12-31,Zwolle,Utrecht Centraal,23,104,53.608696,40,226,15.375000,5,...,False,12,308,0.422827,308,72843,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,2024-12-31,Winter


In [4]:
import networkx as nx

# Ensure the 'Date' column is in datetime format
disruptions_df['Date'] = pd.to_datetime(disruptions_df['Date'])

# Get unique dates and stations
unique_dates = disruptions_df['Date'].dt.date.unique()
sources = disruptions_df['source'].unique()
destinations = disruptions_df['destination'].unique()
stations = list(set(sources).union(set(destinations)))

# Function to create daily graph and calculate node features
def create_daily_graph(date, df):
    # Filter data for the specific date
    daily_df = df[df['Date'].dt.date == date]
    
    # Create graph
    G = nx.DiGraph()
    G.add_nodes_from(stations)
    
    # Add weighted edges
    for _, row in daily_df.iterrows():
        G.add_edge(
            row['source'], 
            row['destination'], 
            weight=row['weight'], 
            distance=row['distance']
        )
        # print(f"Added edge from {row['source']} to {row['target']} with weight {row['Rides planned']} and distance {row['distance']}")
    
    # Calculate node features
    node_features = {}
    for node in G.nodes():
        # Degree (number of connections)
        degree = G.degree(node)
        
        # Weighted degree (sum of rides planned)
        weighted_degree = sum(
            G[node][neighbor]['weight'] 
            for neighbor in G.neighbors(node)
        )
        
        # Average distance to neighbors
        avg_distance = (
            sum(G[node][neighbor]['distance'] for neighbor in G.neighbors(node)) / degree
            if degree > 0 else 0
        )
        
        node_features[node] = {
            'degree': degree,
            'weighted_degree': weighted_degree,
            'avg_distance': avg_distance
        }
    # Calculate edge features
    edge_features = {}
    for edge in G.edges():
        source, target = edge
        row = daily_df[(daily_df['source'] == source) & (daily_df['destination'] == target)]
        if not row.empty:
            edge_features[edge] = row.iloc[0].drop(['Date', 'source', 'destination', 'Disrupted', 'Total_disruptions', 'Total_rides', 'Previous_causes_vec', 'Most_recent_causes_vec', 'Date.1', 'Season']).to_dict()
        else:
            edge_features[edge] = {}  # Handle cases where no matching row is found
    
    return G, node_features, edge_features

example_date = unique_dates[0]
G, node_features, edge_features = create_daily_graph(example_date, disruptions_df)
daily_df = disruptions_df[disruptions_df['Date'].dt.date == example_date]

In [5]:
print(edge_features)

{('Rotterdam Centraal', 'Amsterdam Centraal'): {'source_degree': 21, 'source_weighted_degree': 134, 'source_avg_distance': 53.0, 'target_degree': 49, 'target_weighted_degree': 437, 'target_avg_distance': 41.30612244897959, 'common_neighbors': 8, 'jaccard_coefficient': 0.25, 'preferential_attachment': 1029, 'adamic_adar_index': 5.091285633253196, 'resource_allocation_index': 1.5121212121212122, 'weight': 36, 'distance': 86.0, 'RH': 9.0, 'SQ': 9.0, 'TG': 78.0, 'TN': 58.0, 'TX': 91.0, 'RHX': 3.0, 'VVX': 76.0, 'T10N': 50.0, 'FHVEC': 52.0, 'Days_since_last_disruption': 0, 'Num_prev_disruptions': 0, 'Ratio': 0.0}, ('Eindhoven Centraal', 'Den Haag Centraal'): {'source_degree': 21, 'source_weighted_degree': 116, 'source_avg_distance': 40.142857142857146, 'target_degree': 29, 'target_weighted_degree': 253, 'target_avg_distance': 52.793103448275865, 'common_neighbors': 3, 'jaccard_coefficient': 0.1111111111111111, 'preferential_attachment': 609, 'adamic_adar_index': 1.3050640741175197, 'resource

In [6]:
edge_features_list = [
            'source_degree', 'source_weighted_degree', 'source_avg_distance', 'target_degree', 'target_weighted_degree', 'target_avg_distance', 'common_neighbors', 'jaccard_coefficient',
             'preferential_attachment', 'adamic_adar_index', 'resource_allocation_index', 'weight', 'distance', 'RH', 'SQ', 'TG', 'TN', 'TX', 'RHX', 'VVX', 'T10N', 'FHVEC', 
            'Days_since_last_disruption', 'Num_prev_disruptions', 'Ratio'
            ]

In [7]:
# Dictionary to store daily graphs and features
daily_graphs = {}
daily_features = {}
daily_edge_features = {}

# Process each day
for date in unique_dates:
    G, features, edge_features = create_daily_graph(date, disruptions_df)
    daily_graphs[date] = G
    daily_features[date] = features
    daily_edge_features[date] = edge_features

In [8]:
trajectories_dict = {}
for date in unique_dates:
    daily_df = disruptions_df[disruptions_df['Date'].dt.date == date]
    G = daily_graphs[date]
    trajectories_dict[date] = []  # Initialize an empty list for each date
    for edge in G.edges():
        source, target = edge
        disrupted = daily_df[(daily_df['source'] == source) & (daily_df['destination'] == target)]['Disrupted'].values
        disrupted = disrupted[0] if len(disrupted) > 0 else False  # Handle cases where no match is found
        trajectories_dict[date].append((edge, disrupted))

In [9]:
from datetime import datetime

def get_season(date):
    year = date.year
    date = datetime(date.year, date.month, date.day)  # Normalize to remove time component
    spring_start = datetime(year, 3, 21)
    summer_start = datetime(year, 6, 21)
    autumn_start = datetime(year, 9, 21)
    winter_start = datetime(year, 12, 21)

    if date >= winter_start or date < spring_start:
        return 'Winter'
    elif spring_start <= date < summer_start:
        return 'Spring'
    elif summer_start <= date < autumn_start:
        return 'Summer'
    elif autumn_start <= date < winter_start:
        return 'Autumn'

In [10]:
from torch_geometric.data import Data
import torch

# Function to convert NetworkX graph and features to Data object
def graph_to_data(date, G, node_features, edge_features, trajectories):
    nodes = list(G.nodes())
    x = torch.tensor([list(node_features[node].values()) for node in nodes], dtype=torch.float)

    edges = list(G.edges())
    edge_index = torch.tensor([[nodes.index(u), nodes.index(v)] for u, v in edges], dtype=torch.long).t()
    edge_attr = torch.tensor([
        list(edge_features.get((u, v), {k: 0 for k in next(iter(edge_features.values())).keys()}).values())
        for u, v in edges
    ], dtype=torch.float)

    season = get_season(date)
    year = date.year

    data_list = []
    for traj, label in trajectories:
        traj_indices = torch.tensor([nodes.index(node) for node in traj], dtype=torch.long)
        data = Data(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            trajectory=torch.tensor([nodes.index(node) for node in traj], dtype=torch.long),
            y=torch.tensor(label, dtype=torch.float)  # Not wrapped in [label]
        )
        # Add metadata for splitting
        data.season = season
        data.year = year
        data_list.append(data)

    return data_list

In [35]:
def get_trajectory_edge_attr(edge_index, edge_attr, trajectory):
    # trajectory: [T] (node indices)
    traj_edge_indices = torch.stack([trajectory[:-1], trajectory[1:]], dim=0)  # [2, T-1]
    # Find matching edges in edge_index
    traj_edge_attr = []
    for i in range(traj_edge_indices.shape[1]):
        src, dst = traj_edge_indices[0, i], traj_edge_indices[1, i]
        # Find edge (src, dst) or (dst, src) in edge_index
        mask = ((edge_index[0] == src) & (edge_index[1] == dst)) | ((edge_index[0] == dst) & (edge_index[1] == src))
        edge_idx = torch.where(mask)[0]
        if edge_idx.numel() > 0:
            traj_edge_attr.append(edge_attr[edge_idx[0]])
        else:
            # Handle missing edges (e.g., use zero vector or mean edge_attr)
            traj_edge_attr.append(torch.zeros_like(edge_attr[0]))
    return torch.stack(traj_edge_attr)  # [T-1, edge_feature_dim]

In [56]:
import torch.nn as nn
from torch_geometric.nn import GATConv, global_mean_pool
import torch.nn.functional as F
from torch_geometric.data import Data


class GNN_LSTM(nn.Module):
    def __init__(self, node_feature_dim, edge_feature_dim, hidden_dim, output_dim):
        super().__init__()
        # GATConv with multi-head attention
        self.conv1 = GATConv(node_feature_dim, hidden_dim, heads=4, edge_dim=edge_feature_dim)
        self.conv2 = GATConv(hidden_dim * 4, hidden_dim, heads=1, edge_dim=edge_feature_dim)
        self.skip1 = nn.Linear(node_feature_dim, hidden_dim * 4)  # Residual connection
        self.bn1 = nn.BatchNorm1d(hidden_dim * 4)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim // 2, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_dim + edge_feature_dim, output_dim)

    def forward(self, data):
        x, edge_index, edge_attr, trajectory = data.x, data.edge_index, data.edge_attr, data.trajectory
        
        # Normalize features
        x = (x - x.mean(dim=0)) / (x.std(dim=0) + 1e-8)
        edge_attr = (edge_attr - edge_attr.mean(dim=0)) / (edge_attr.std(dim=0) + 1e-8)

        # GATConv layers with residual connection
        x_in = x
        x = self.conv1(x, edge_index, edge_attr)
        x = self.bn1(x + self.skip1(x_in))
        x = torch.relu(x)
        x = self.bn2(self.conv2(x, edge_index, edge_attr))
        x = torch.relu(x)
        x = F.dropout(x, p=0.35, training=self.training)

        # Extract trajectory embeddings
        traj_embeddings = x[trajectory]  # [T, hidden_dim]

        # LSTM for sequential modeling
        lstm_out, _ = self.lstm(traj_embeddings.unsqueeze(0))  # [1, T, hidden_dim//2]
        # Edge classification (consecutive nodes in trajectory)
        edge_embeddings = torch.cat([lstm_out[:, :-1], lstm_out[:, 1:]], dim=-1)  # [1, T-1, hidden_dim]
        traj_edge_features = get_trajectory_edge_attr(edge_index, edge_attr, trajectory)  # [T-1, edge_feature_dim]
        edge_input = torch.cat([edge_embeddings.squeeze(0), traj_edge_features], dim=-1)  # [T-1, hidden_dim + edge_feature_dim]
        out = self.fc(edge_input)  # [T-1, output_dim]
        return out

In [12]:
from collections import defaultdict
seasonal_dataset = defaultdict(list)  # {'Winter': [Data, Data, ...], ...}

for date in unique_dates:
    G = daily_graphs[date]
    node_features = daily_features[date]
    edge_features = daily_edge_features[date]
    trajectories = trajectories_dict[date]
    
    daily_data = graph_to_data(date, G, node_features, edge_features, trajectories)
    season = get_season(date)
    if season is None:
        print(date)
    for data in daily_data:
        seasonal_dataset[season].append(data)

# (Optional) Check size of each seasonal split
for season, data_list in seasonal_dataset.items():
    print(f"{season}: {len(data_list)} samples")

Winter: 135213 samples
Spring: 145919 samples
Summer: 147230 samples
Autumn: 146791 samples


In [14]:
from torch_geometric.loader import DataLoader

def split_by_year(seasonal_data, test_year=2024):
    train_data = [d for d in seasonal_data if d.year != test_year]
    test_data = [d for d in seasonal_data if d.year == test_year]
    return train_data, test_data

# For Winter 2024 test
winter_train, winter_test = split_by_year(seasonal_dataset['Winter'], test_year=2024)
train_loader = DataLoader(winter_train, batch_size=32, shuffle=True)
test_loader = DataLoader(winter_test, batch_size=32, shuffle=False)

In [62]:
from sklearn.metrics import precision_score, recall_score, f1_score, balanced_accuracy_score
from torch.optim.lr_scheduler import ReduceLROnPlateau

models = {}
for season in ['Winter', 'Spring', 'Summer', 'Autumn']:
    print(f"Processing season: {season}")
    X_train, X_test = split_by_year(seasonal_dataset[season], test_year=2024)
    print(f"{season} — Test set size: {len(X_test)}")

    # Initialize model, loss, and optimizer
    node_feature_dim = 3
    edge_feature_dim = 25
    hidden_dim = 32
    output_dim = 1
    model = GNN_LSTM(node_feature_dim, edge_feature_dim, hidden_dim, output_dim)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([0.86 / 0.14]))
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
    train_loader = DataLoader(X_train, batch_size=1, shuffle=False)
    test_loader = DataLoader(X_test, batch_size=1, shuffle=False)

    # Training loop
    def train():
        model.train()
        total_loss = 0
        for data in train_loader:
            optimizer.zero_grad()
            out = model(data)  # [T-1, output_dim]
            y = data.y  # [T-1]
            loss = criterion(out.squeeze(-1), y.float())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        return total_loss / len(train_loader)

    # Evaluation loop
    def test():
        model.eval()
        total_loss = 0
        y_true = []
        y_pred = []

        with torch.no_grad():
            for data in test_loader:
                out = model(data)  # [T-1, output_dim]
                y = data.y  # [T-1]
                loss = criterion(out.squeeze(-1), y.float())
                total_loss += loss.item()

                preds = (torch.sigmoid(out.squeeze(-1)) >= 0.5).int().cpu().numpy()
                labels = y.int().cpu().numpy()

                y_pred.extend(preds)
                y_true.extend(labels)

        precision = precision_score(y_true, y_pred)
        recall = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        ba = balanced_accuracy_score(y_true, y_pred)
        avg_loss = total_loss / len(test_loader)
        return avg_loss, precision, recall, f1, ba

    # Early stopping parameters
    patience = 5
    best_test_loss = np.inf
    patience_counter = 0
    best_model_path = f"best_gnn_lstm_model_{season}.pth"

    epochs = 50
    for epoch in range(epochs):
        train_loss = train()
        test_loss, p, r, f1, ba = test()
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}, F1: {f1:.4f}")

        scheduler.step(test_loss)

        if test_loss < best_test_loss:
            best_test_loss = test_loss
            patience_counter = 0
            torch.save(model.state_dict(), best_model_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                print(f"Best Test Loss: {best_test_loss:.4f}, Precision: {p:.4f}, Recall: {r:.4f}, F1: {f1:.4f}, Balanced Accuracy: {ba:.4f}")
                break

    # Load the best model
    model.load_state_dict(torch.load(best_model_path))
    models[season] = model
    print("Loaded best model with test loss: {:.4f}".format(best_test_loss))

Processing season: Winter
Winter — Test set size: 24608
Epoch 1, Train Loss: 0.9201, Test Loss: 1.4081, F1: 0.2041
Epoch 2, Train Loss: 0.9206, Test Loss: 1.1751, F1: 0.3340
Epoch 3, Train Loss: 0.9236, Test Loss: 1.3947, F1: 0.1899
Epoch 4, Train Loss: 0.9227, Test Loss: 1.3754, F1: 0.1978
Epoch 5, Train Loss: 0.9219, Test Loss: 1.4930, F1: 0.1745
Epoch 6, Train Loss: 0.9223, Test Loss: 1.3603, F1: 0.2123
Epoch 7, Train Loss: 0.9314, Test Loss: 1.1545, F1: 0.3354
Epoch 8, Train Loss: 0.9298, Test Loss: 1.2243, F1: 0.2907
Epoch 9, Train Loss: 0.9289, Test Loss: 1.2255, F1: 0.2800
Epoch 10, Train Loss: 0.9273, Test Loss: 1.0677, F1: 0.3758
Epoch 11, Train Loss: 0.9283, Test Loss: 1.0869, F1: 0.3634
Epoch 12, Train Loss: 0.9292, Test Loss: 1.1060, F1: 0.3549
Epoch 13, Train Loss: 0.9294, Test Loss: 1.1127, F1: 0.3483
Epoch 14, Train Loss: 0.9283, Test Loss: 1.1567, F1: 0.3399
Epoch 15, Train Loss: 0.9378, Test Loss: 1.0862, F1: 0.3582
Early stopping triggered after 15 epochs.
Best Test L

c:\Users\brake\UVA_AML24\week_1\.conda\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1, Train Loss: 0.9488, Test Loss: 0.9814, F1: 0.3018
Epoch 2, Train Loss: 0.9443, Test Loss: 0.9830, F1: 0.3004
Epoch 3, Train Loss: 0.9422, Test Loss: 0.9975, F1: 0.2949
Epoch 4, Train Loss: 0.9412, Test Loss: 0.9918, F1: 0.2960
Epoch 5, Train Loss: 0.9452, Test Loss: 0.9935, F1: 0.2969
Epoch 6, Train Loss: 0.9546, Test Loss: 0.9721, F1: 0.3033
Epoch 7, Train Loss: 0.9493, Test Loss: 0.9808, F1: 0.3023
Epoch 8, Train Loss: 0.9491, Test Loss: 0.9799, F1: 0.3038
Epoch 9, Train Loss: 0.9517, Test Loss: 0.9807, F1: 0.3038
Epoch 10, Train Loss: 0.9465, Test Loss: 0.9768, F1: 0.3026
Epoch 11, Train Loss: 0.9654, Test Loss: 0.9644, F1: 0.3031
Epoch 12, Train Loss: 0.9647, Test Loss: 0.9680, F1: 0.3005
Epoch 13, Train Loss: 0.9563, Test Loss: 0.9541, F1: 0.3072
Epoch 14, Train Loss: 0.9570, Test Loss: 0.9651, F1: 0.3041
Epoch 15, Train Loss: 0.9541, Test Loss: 0.9659, F1: 0.3025
Epoch 16, Train Loss: 0.9596, Test Loss: 0.9576, F1: 0.3059
Epoch 17, Train Loss: 0.9578, Test Loss: 0.9596, 

c:\Users\brake\UVA_AML24\week_1\.conda\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1, Train Loss: 0.9256, Test Loss: 0.9245, F1: 0.3411
Epoch 2, Train Loss: 0.9297, Test Loss: 0.9569, F1: 0.3335
Epoch 3, Train Loss: 0.9260, Test Loss: 0.9345, F1: 0.3388
Epoch 4, Train Loss: 0.9257, Test Loss: 0.9484, F1: 0.3326
Epoch 5, Train Loss: 0.9247, Test Loss: 1.0161, F1: 0.3053
Epoch 6, Train Loss: 0.9275, Test Loss: 0.9270, F1: 0.3410
Early stopping triggered after 6 epochs.
Best Test Loss: 0.9245, Precision: 0.2443, Recall: 0.5643, F1: 0.3410, Balanced Accuracy: 0.6727
Loaded best model with test loss: 0.9245
Processing season: Autumn
Autumn — Test set size: 24806


c:\Users\brake\UVA_AML24\week_1\.conda\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1, Train Loss: 0.9740, Test Loss: 0.9230, F1: 0.3788
Epoch 2, Train Loss: 0.9730, Test Loss: 0.9257, F1: 0.3840
Epoch 3, Train Loss: 0.9724, Test Loss: 0.9215, F1: 0.3838
Epoch 4, Train Loss: 0.9719, Test Loss: 0.9306, F1: 0.3790
Epoch 5, Train Loss: 0.9694, Test Loss: 0.9184, F1: 0.3849
Epoch 6, Train Loss: 0.9686, Test Loss: 0.9193, F1: 0.3793
Epoch 7, Train Loss: 0.9691, Test Loss: 0.9158, F1: 0.3847
Epoch 8, Train Loss: 0.9699, Test Loss: 0.9191, F1: 0.3839
Epoch 9, Train Loss: 0.9698, Test Loss: 0.9193, F1: 0.3803
Epoch 10, Train Loss: 0.9702, Test Loss: 0.9242, F1: 0.3823
Epoch 11, Train Loss: 0.9691, Test Loss: 0.9290, F1: 0.3822
Epoch 12, Train Loss: 0.9712, Test Loss: 0.9125, F1: 0.3797
Epoch 13, Train Loss: 0.9694, Test Loss: 0.9144, F1: 0.3831
Epoch 14, Train Loss: 0.9667, Test Loss: 0.9131, F1: 0.3806
Epoch 15, Train Loss: 0.9663, Test Loss: 0.9132, F1: 0.3803
Epoch 16, Train Loss: 0.9673, Test Loss: 0.9116, F1: 0.3821
Epoch 17, Train Loss: 0.9706, Test Loss: 0.9127, 